# Feature Engineering Basics

Feature engineering is the process of transforming raw columns into inputs that make structure easier for models to learn. In this notebook, we'll focus on the parts that go beyond basic scaling:

1. **One-Hot Encoding** - Converting categorical variables into binary vectors
2. **Binning** - Grouping continuous values into discrete intervals
3. **Feature Pipelines** - Combining multiple transformations into a final modeling matrix

We'll assume you already know the basics of normalization and standardization from the dedicated `data-normalization` notebook, and we'll reuse scaling only inside the final end-to-end pipeline.

## Setup

Import the libraries we'll need for our feature engineering exploration.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Set random seed for reproducibility
np.random.seed(42)

## A Quick Prerequisite Note

**Scaling recap**: normalization and standardization are foundational preprocessing tools, but they are already covered in detail in `data-normalization.ipynb`.

Here we'll focus on transformations that change feature meaning or structure: categorical encoding, discretization, and combining multiple preprocessing steps into a final feature matrix.

# 1. One-Hot Encoding

## The Categorical Data Problem

Machine learning algorithms work with numbers, but real-world data often contains categories:
- Colors: Red, Blue, Green
- Cities: New York, London, Tokyo
- Product Types: Electronics, Clothing, Food

Simply assigning numbers (Red=1, Blue=2, Green=3) is problematic because it implies ordering and distance that don't exist. One-hot encoding solves this elegantly.

## How One-Hot Encoding Works

For each unique category, create a new binary (0/1) column:

| Original | → | is_Red | is_Blue | is_Green |
|----------|---|--------|---------|----------|
| Red      | → |   1    |    0    |    0     |
| Blue     | → |   0    |    1    |    0     |
| Green    | → |   0    |    0    |    1     |

Each row has exactly one '1' (hence "one-hot"), indicating which category it belongs to.

## Simple Example: Fruit Types

Let's encode fruit types into one-hot vectors.

In [ ]:
# Sample fruit data
fruits = np.array(['Apple', 'Banana', 'Apple', 'Cherry', 'Banana', 'Cherry', 'Apple'])

print("Original categorical data:")
print(fruits)
print(f"\nUnique categories: {np.unique(fruits)}")

Manual one-hot encoding to understand the concept.

In [ ]:
# Manual one-hot encoding
unique_fruits = np.unique(fruits)
one_hot_manual = np.zeros((len(fruits), len(unique_fruits)), dtype=int)

for i, fruit in enumerate(fruits):
    col_idx = np.where(unique_fruits == fruit)[0][0]
    one_hot_manual[i, col_idx] = 1

one_hot_df = pd.DataFrame(one_hot_manual, columns=[f'is_{fruit}' for fruit in unique_fruits])
one_hot_df.insert(0, 'Original', fruits)

print("One-Hot Encoded:")
print(one_hot_df)

## Using pandas get_dummies

Pandas provides a convenient function for one-hot encoding.

In [ ]:
# Create a simple DataFrame
df = pd.DataFrame({
    'fruit': ['Apple', 'Banana', 'Cherry'],
    'color': ['Red', 'Yellow', 'Red'],
    'price': [1.2, 0.5, 2.0]
})

print("Original DataFrame:")
print(df)

Apply get_dummies to encode all categorical columns at once.

In [ ]:
# One-hot encode with pandas
df_encoded = pd.get_dummies(df, columns=['fruit', 'color'])

print("\nOne-Hot Encoded DataFrame:")
print(df_encoded)

## Using sklearn OneHotEncoder

For production pipelines, sklearn's encoder is more robust and handles new categories.

In [ ]:
# Training data
train_colors = np.array(['Red', 'Blue', 'Green', 'Red', 'Blue']).reshape(-1, 1)

# Fit encoder on training data
encoder = OneHotEncoder(sparse_output=False)
encoder.fit(train_colors)

print("Categories learned:")
print(encoder.categories_[0])
print(f"\nFeature names:")
print(encoder.get_feature_names_out())

Transform both training and new test data using the fitted encoder.

In [ ]:
# Transform training data
train_encoded = encoder.transform(train_colors)

print("Training data encoded:")
train_df = pd.DataFrame(train_encoded, columns=encoder.get_feature_names_out())
train_df.insert(0, 'Original', train_colors.flatten())
print(train_df)

## Handling Unknown Categories

What happens when test data contains categories not seen during training?

In [ ]:
# Create encoder that handles unknown categories
encoder_safe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder_safe.fit(train_colors)

# Test data with unknown category
test_colors = np.array(['Red', 'Yellow', 'Blue']).reshape(-1, 1)  # 'Yellow' is new!
test_encoded = encoder_safe.transform(test_colors)

print("Test data with unknown category:")
test_df = pd.DataFrame(test_encoded, columns=encoder_safe.get_feature_names_out())
test_df.insert(0, 'Original', test_colors.flatten())
print(test_df)
print("\nNote: 'Yellow' (unknown) is encoded as all zeros")

## Visualizing One-Hot Encoding

See the transformation from categorical to binary representation visually.

In [ ]:
# Create sample data for visualization
categories = ['Cat', 'Dog', 'Bird', 'Cat', 'Dog', 'Cat', 'Bird', 'Dog']
cat_encoded = pd.get_dummies(pd.Series(categories))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Original categories as numbers (wrong approach)
cat_numbers = [0 if c == 'Cat' else 1 if c == 'Dog' else 2 for c in categories]
ax1.scatter(range(len(cat_numbers)), cat_numbers, s=200, c='red', alpha=0.6)
ax1.set_yticks([0, 1, 2])
ax1.set_yticklabels(['Cat', 'Dog', 'Bird'])
ax1.set_xlabel('Sample Index')
ax1.set_title('❌ Wrong: Ordinal Encoding\n(implies Dog > Cat > Bird)')
ax1.grid(True, alpha=0.3)

# One-hot encoding (correct approach)
sns.heatmap(cat_encoded.T, annot=True, fmt='d', cmap='RdYlGn', cbar=False, ax=ax2)
ax2.set_xlabel('Sample Index')
ax2.set_ylabel('Category')
ax2.set_title('✓ Correct: One-Hot Encoding\n(no implied ordering)')

plt.tight_layout()
plt.show()

## The Dummy Variable Trap

When using linear models, you can drop one category to avoid multicollinearity (the dummy variable trap). If you know all other categories are 0, you automatically know the dropped category is 1.

In [ ]:
# With drop_first=True, one category is dropped
df_dummy_trap = pd.get_dummies(pd.Series(['Red', 'Blue', 'Green']), drop_first=True)

print("One-Hot with drop_first=True:")
print(df_dummy_trap)
print("\nRed is the reference category (all zeros means Red)")

## When to Use One-Hot Encoding

**Best for:**
- Nominal categories (no natural order): colors, cities, product types
- Small number of unique categories (< 10-20)
- Tree-based models (Random Forest, XGBoost)
- Neural networks

**Limitations:**
- High cardinality (many unique values) creates too many columns
- Sparse data (mostly zeros) increases memory usage

**Alternatives for high cardinality:**
- Target encoding
- Embeddings (for neural networks)

# 2. Binning (Discretization)

## Why Bin Continuous Data?

Binning groups continuous values into discrete intervals. This can:
- Reduce the impact of minor observation errors
- Handle outliers more gracefully
- Introduce non-linearity to linear models
- Make relationships easier to interpret

Example: Converting exact ages (23, 45, 67) into age groups (18-30, 31-50, 51-70).

## Two Main Binning Strategies

1. **Equal-Width Binning**: Divide the range into bins of equal size
   - Example: 0-25, 25-50, 50-75, 75-100

2. **Equal-Frequency Binning**: Divide data so each bin has roughly the same number of samples
   - Example: First quartile, second quartile, etc.

Each has different use cases depending on your data distribution.

## Example Dataset: Customer Ages

Create a sample dataset of customer ages to demonstrate binning.

In [ ]:
# Generate sample age data (skewed distribution)
np.random.seed(42)
ages = np.concatenate([
    np.random.normal(25, 3, 40),   # Young adults
    np.random.normal(45, 5, 30),   # Middle-aged
    np.random.normal(70, 4, 20)    # Seniors
])
ages = np.clip(ages, 18, 85)  # Realistic age bounds

print(f"Number of samples: {len(ages)}")
print(f"Age range: {ages.min():.1f} - {ages.max():.1f}")
print(f"Mean age: {ages.mean():.1f}")

Visualize the distribution of ages to understand what we're working with.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(ages, bins=20, color='steelblue', alpha=0.7, edgecolor='black')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.title('Distribution of Customer Ages')
plt.axvline(ages.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean = {ages.mean():.1f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Equal-Width Binning

Divide the age range into bins of equal width using pandas.cut().

In [ ]:
# Equal-width binning into 4 bins
ages_binned_width = pd.cut(ages, bins=4)

print("Equal-Width Bins:")
print(ages_binned_width.value_counts().sort_index())
print(f"\nBin intervals: {ages_binned_width.categories.tolist()}")

Apply custom labels to make the bins more interpretable.

In [ ]:
# Custom bins with labels
bins = [18, 30, 45, 60, 85]
labels = ['Young Adult', 'Adult', 'Middle-Aged', 'Senior']
ages_categorized = pd.cut(ages, bins=bins, labels=labels)

print("Custom Age Categories:")
print(ages_categorized.value_counts())

## Equal-Frequency Binning (Quantiles)

Divide data into bins with approximately equal number of samples using pandas.qcut().

In [ ]:
# Equal-frequency binning (quartiles)
ages_binned_freq = pd.qcut(ages, q=4)

print("Equal-Frequency Bins (Quartiles):")
print(ages_binned_freq.value_counts().sort_index())
print(f"\nNote: Each bin has approximately {len(ages)//4} samples")

Apply custom labels to quantile bins.

In [ ]:
# Quantile binning with labels
ages_quantiles = pd.qcut(ages, q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

print("Age Quartiles:")
print(ages_quantiles.value_counts())

## Comparing Binning Strategies

Visualize how equal-width and equal-frequency binning differ on the same data.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Equal-width binning visualization
ages_width_codes = pd.cut(ages, bins=4).codes
ax1.hist(ages, bins=20, alpha=0.3, color='gray', label='Original distribution')
for i, interval in enumerate(pd.cut(ages, bins=4).categories):
    mask = ages_width_codes == i
    ax1.hist(ages[mask], bins=20, alpha=0.6, label=f'Bin {i+1}')
ax1.set_xlabel('Age')
ax1.set_ylabel('Frequency')
ax1.set_title('Equal-Width Binning\n(Bins have equal width but different counts)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Equal-frequency binning visualization
ages_freq_codes = pd.qcut(ages, q=4).codes
ax2.hist(ages, bins=20, alpha=0.3, color='gray', label='Original distribution')
for i, interval in enumerate(pd.qcut(ages, q=4).categories):
    mask = ages_freq_codes == i
    ax2.hist(ages[mask], bins=20, alpha=0.6, label=f'Bin {i+1}')
ax2.set_xlabel('Age')
ax2.set_ylabel('Frequency')
ax2.set_title('Equal-Frequency Binning\n(Bins have different widths but similar counts)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Create a detailed comparison table showing the characteristics of each binning strategy.

In [ ]:
# Comparison statistics
width_bins = pd.cut(ages, bins=4)
freq_bins = pd.qcut(ages, q=4)

comparison_df = pd.DataFrame({
    'Strategy': ['Equal-Width'] * 4 + ['Equal-Frequency'] * 4,
    'Bin': list(width_bins.categories) + list(freq_bins.categories),
    'Count': list(width_bins.value_counts().sort_index()) + list(freq_bins.value_counts().sort_index()),
    'Width': [interval.length for interval in width_bins.categories] + 
             [interval.length for interval in freq_bins.categories]
})

print("Binning Strategy Comparison:")
print(comparison_df.to_string(index=False))

## Converting Bins to Numeric Features

After binning, you can use the bins in different ways depending on your model.

In [ ]:
# Create a DataFrame with original and binned ages
age_df = pd.DataFrame({'age': ages})
age_df['age_category'] = pd.cut(ages, bins=bins, labels=labels)

# Option 1: Use bin labels as ordinal encoding
age_df['age_ordinal'] = age_df['age_category'].cat.codes

# Option 2: One-hot encode the bins
age_one_hot = pd.get_dummies(age_df['age_category'], prefix='age')
age_df_encoded = pd.concat([age_df, age_one_hot], axis=1)

print("Sample of encoded age data:")
print(age_df_encoded.head(10))

## When to Use Binning

**Equal-Width Binning:**
- When you want intuitive, evenly-spaced intervals
- For visualizations and reporting
- When data is uniformly distributed

**Equal-Frequency Binning:**
- When data is skewed or has outliers
- When you want balanced classes
- For percentile-based analysis

**General use cases:**
- Converting continuous features for decision trees
- Creating age/income groups for analysis
- Handling non-linear relationships in linear models
- Reducing the impact of outliers

# 3. Putting It All Together

## Real-World Feature Engineering Pipeline

Let's apply multiple techniques to a realistic dataset. We'll create synthetic customer data and prepare it for machine learning.

In [ ]:
# Create synthetic customer dataset
np.random.seed(42)
n_customers = 100

customer_data = pd.DataFrame({
    'age': np.random.randint(18, 75, n_customers),
    'income': np.random.normal(50000, 20000, n_customers),
    'spending_score': np.random.randint(1, 100, n_customers),
    'country': np.random.choice(['USA', 'UK', 'Germany', 'France'], n_customers),
    'membership': np.random.choice(['Bronze', 'Silver', 'Gold'], n_customers)
})

# Clip income to realistic values
customer_data['income'] = customer_data['income'].clip(20000, 150000)

print("Raw Customer Data (first 5 rows):")
print(customer_data.head())
print(f"\nShape: {customer_data.shape}")
print(f"\nData types:\n{customer_data.dtypes}")

## Step 1: Bin the Age Feature

Convert continuous age into meaningful age groups.

In [ ]:
# Bin age into groups
age_bins = [18, 30, 45, 60, 75]
age_labels = ['Young', 'Adult', 'Middle-Aged', 'Senior']
customer_data['age_group'] = pd.cut(customer_data['age'], bins=age_bins, labels=age_labels)

print("Age group distribution:")
print(customer_data['age_group'].value_counts().sort_index())

## Step 2: Scale Numerical Features

Use one consistent scaler so numerical features live on comparable ranges.

In [ ]:
# Select numerical features for scaling
numerical_features = ['income', 'spending_score']

# Scale using MinMaxScaler
scaler = MinMaxScaler()
customer_data[['income_norm', 'spending_norm']] = scaler.fit_transform(customer_data[numerical_features])

print("Before and after scaling:")
print(customer_data[['income', 'income_norm', 'spending_score', 'spending_norm']].head())

## Step 3: One-Hot Encode Categorical Features

Convert country and age_group into binary features.

In [ ]:
# One-hot encode categorical features
customer_encoded = pd.get_dummies(
    customer_data,
    columns=['country', 'age_group', 'membership'],
    prefix=['country', 'age', 'member']
)

print(f"Shape after encoding: {customer_encoded.shape}")
print(f"\nNew columns created:")
new_cols = [col for col in customer_encoded.columns if col not in customer_data.columns]
print(new_cols)

## Step 4: Create Final Feature Matrix

Select only the engineered features for modeling, dropping original raw features.

In [ ]:
# Select final features for modeling
feature_columns = ['income_norm', 'spending_norm'] + new_cols
X = customer_encoded[feature_columns]

print("Final feature matrix:")
print(X.head())
print(f"\nFinal shape: {X.shape}")
print(f"Features: {X.columns.tolist()}")

## Visualize the Feature Engineering Pipeline

Compare the raw data distribution with the engineered features.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Original income distribution
axes[0, 0].hist(customer_data['income'], bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Income ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Original Income Distribution')

# Normalized income distribution
axes[0, 1].hist(customer_data['income_norm'], bins=20, color='coral', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Normalized Income')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Normalized Income [0, 1]')

# Age group distribution
age_counts = customer_data['age_group'].value_counts().sort_index()
axes[1, 0].bar(range(len(age_counts)), age_counts.values, color='green', alpha=0.7)
axes[1, 0].set_xticks(range(len(age_counts)))
axes[1, 0].set_xticklabels(age_counts.index, rotation=45)
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Age Groups (After Binning)')

# Country one-hot encoding sample
country_cols = [col for col in X.columns if col.startswith('country_')]
sample_countries = X[country_cols].head(10)
sns.heatmap(sample_countries.T, annot=True, fmt='d', cmap='RdYlGn', cbar=False, ax=axes[1, 1])
axes[1, 1].set_xlabel('Customer Index')
axes[1, 1].set_title('Country One-Hot Encoding (First 10 Customers)')

plt.tight_layout()
plt.show()

## Summary Statistics

Compare the feature space before and after engineering.

In [ ]:
print("=" * 60)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 60)
print(f"\nOriginal dataset:")
print(f"  - Shape: {customer_data.shape}")
print(f"  - Numerical features: {len(numerical_features)}")
print(f"  - Categorical features: 3 (country, age_group, membership)")
print(f"\nEngineered dataset:")
print(f"  - Shape: {X.shape}")
print(f"  - All features normalized/encoded and ready for ML")
print(f"  - Features created: {X.shape[1]}")
print(f"\nTransformations applied:")
print(f"  1. Binning: age → age_group (4 categories)")
print(f"  2. Normalization: income, spending_score → [0, 1] range")
print(f"  3. One-hot encoding: country, age_group, membership → binary features")
print("\n" + "=" * 60)

# Key Takeaways

## What This Notebook Added

| Technique | Use Case | Best For |
|-----------|----------|----------|
| **One-Hot Encoding** | Categorical → Binary | Nominal categories, tree-based models, neural networks |
| **Binning** | Continuous → Discrete | Age/income groups, handling outliers, creating non-linearity |
| **Feature Pipelines** | Combining multiple transformations | Real-world tabular preprocessing |
| **Scaling (used here, covered separately)** | Put numeric features on comparable ranges | Distance-based models, neural nets, stable optimization |

## Feature Engineering Best Practices

1. **Always split first**: Fit transformations on training data only, then apply to validation/test data
2. **Handle missing values explicitly**: Impute or drop before encoding and scaling
3. **Choose transformations for the model**: Different algorithms benefit from different representations
4. **Keep features interpretable**: Engineered columns should still make sense to a human reader
5. **Pipeline it**: Use sklearn's `Pipeline` and `ColumnTransformer` for reproducible preprocessing
6. **Monitor distributions**: Visualize before and after transformations

## Common Pitfalls to Avoid

- ❌ Fitting encoders or scalers on the full dataset (data leakage)
- ❌ One-hot encoding high-cardinality features without thinking about dimensionality
- ❌ Binning without checking where the cut points fall
- ❌ Forgetting to handle unknown categories at inference time
- ❌ Mixing raw and engineered columns without documenting why

Feature engineering is most useful when each transformation has a clear purpose. Prefer a few deliberate features over a large pile of weak ones.